# LeetCode 1674 — Minimum Moves to Make Array Complementary bằng Python

Link đề: https://leetcode.com/problems/minimum-moves-to-make-array-complementary/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Prefix Sum

Skeleton LeetCode:

```python
class Solution:
    def minMoves(self, nums: List[int], limit: int) -> int:
        ...
```

Notebook đi từ **thử mọi mảng đích** (vét cạn theo định nghĩa) → **thử mọi tổng đích `T`** và tính chi phí từng cặp → nhận ra chi phí mỗi cặp là **hàm bậc thang 0/1/2** theo `T` → cộng các bậc thang bằng **mảng hiệu (difference array)** `O(n + limit)` → hai cách khác: **sweep line với sự kiện sắp xếp** và **đếm bằng tìm kiếm nhị phân**.

## 1. Mục tiêu học

1. Chuyển bài về **từng cặp đối xứng** `(nums[i], nums[n−1−i])` và một **tổng đích chung** `T`.
2. Phân tích chi phí của một cặp `(a, b)` (với `a ≤ b`) theo `T`: `0` nếu `T = a + b`; `1` nếu `T ∈ [a + 1, b + limit]`; `2` nếu ngoài đoạn đó.
3. Thành thạo **mảng hiệu**: cộng một hằng số lên cả một đoạn trong `O(1)`, rồi lấy tổng tiền tố để đọc giá trị mọi điểm.
4. Biết hai cách thay thế: **sweep line** trên các sự kiện đã sắp xếp (không phụ thuộc `limit`) và **đếm bằng `bisect`**.
5. Tránh các lỗi biên: phạm vi `T ∈ [2, 2·limit]`, kích thước mảng hiệu, quên sắp `a ≤ b`.

## 2. Đề bài & ví dụ

Cho mảng số nguyên `nums` có độ dài **chẵn** `n` và số nguyên `limit`. Mỗi **bước**, bạn có thể thay **một** phần tử bất kỳ của `nums` bằng một số nguyên trong `[1, limit]`.

Mảng `nums` gọi là **bổ sung (complementary)** nếu với mọi chỉ số `i`, `nums[i] + nums[n − 1 − i]` bằng **cùng một số**. Ví dụ `[1,2,3,4]` là bổ sung vì mọi tổng đối xứng đều bằng `5`.

Trả về **số bước ít nhất** để `nums` trở thành bổ sung.

**Ví dụ 1:** `nums = [1,2,4,3]`, `limit = 4` → `1`. Đổi thành `[1,2,2,3]`: các tổng đối xứng đều bằng `4`.

**Ví dụ 2:** `nums = [1,2,2,1]`, `limit = 2` → `2`. Đổi thành `[2,2,2,2]`. Không thể đổi số nào thành `3` vì `3 > limit`.

**Ví dụ 3:** `nums = [1,2,1,2]`, `limit = 2` → `0` (đã bổ sung).

**Ràng buộc:**
- `n == nums.length`, `2 ≤ n ≤ 10⁵`, `n` chẵn.
- `1 ≤ nums[i] ≤ limit ≤ 10⁵`.

In [1]:
import bisect
import random
import time
from itertools import accumulate, product
from typing import List

examples = [([1, 2, 4, 3], 4, 1), ([1, 2, 2, 1], 2, 2), ([1, 2, 1, 2], 2, 0)]

def pairs_of(nums):
    n = len(nums)
    return [(nums[i], nums[n - 1 - i]) for i in range(n // 2)]

def is_complementary(nums):
    return len({a + b for a, b in pairs_of(nums)}) == 1

for nums, limit, exp in examples:
    print(f"nums={nums}, limit={limit}: các cặp {pairs_of(nums)}, tổng {[a + b for a, b in pairs_of(nums)]}, bổ sung? {is_complementary(nums)} -> kỳ vọng {exp}")
print("[1,2,2,3] bổ sung?", is_complementary([1, 2, 2, 3]))

nums=[1, 2, 4, 3], limit=4: các cặp [(1, 3), (2, 4)], tổng [4, 6], bổ sung? False -> kỳ vọng 1
nums=[1, 2, 2, 1], limit=2: các cặp [(1, 1), (2, 2)], tổng [2, 4], bổ sung? False -> kỳ vọng 2
nums=[1, 2, 1, 2], limit=2: các cặp [(1, 2), (2, 1)], tổng [3, 3], bổ sung? True -> kỳ vọng 0
[1,2,2,3] bổ sung? True


## 3. Kiến thức nền

### 3.1 Cặp đối xứng

Phần tử `i` và `n − 1 − i` tạo thành một cặp; có `n / 2` cặp. Điều kiện "bổ sung" chỉ ràng buộc **tổng của từng cặp**, các cặp **độc lập** với nhau (mỗi phần tử thuộc đúng một cặp).

In [2]:
nums = [10, 20, 30, 40, 50, 60]
n = len(nums)
for i in range(n // 2):
    print(f"cặp {i}: nums[{i}]={nums[i]} và nums[{n - 1 - i}]={nums[n - 1 - i]}")

cặp 0: nums[0]=10 và nums[5]=60
cặp 1: nums[1]=20 và nums[4]=50
cặp 2: nums[2]=30 và nums[3]=40


### 3.2 Mảng hiệu (difference array)

Muốn cộng `v` lên mọi vị trí trong đoạn `[l, r]` của mảng `A`: thay vì lặp, ghi `D[l] += v`, `D[r + 1] −= v`. Sau tất cả các thao tác, `A = tổng tiền tố của D`. Mỗi thao tác đoạn chỉ tốn `O(1)`.

In [3]:
size = 10
D = [0] * (size + 1)
def add_range(l, r, v):
    D[l] += v
    D[r + 1] -= v
add_range(2, 5, 3)
add_range(4, 8, 1)
A = list(accumulate(D))[:size]
print("D =", D)
print("A =", A, "| kiểm tra trực tiếp:", [3 * (2 <= i <= 5) + (4 <= i <= 8) for i in range(size)])

D = [0, 0, 3, 0, 1, 0, -3, 0, 0, -1, 0]
A = [0, 0, 3, 3, 4, 4, 1, 1, 1, 0] | kiểm tra trực tiếp: [0, 0, 3, 3, 4, 4, 1, 1, 1, 0]


### 3.3 `itertools.accumulate`

`accumulate(D)` cho dãy tổng tiền tố; `min(accumulate(D[2:]))` đọc giá trị nhỏ nhất trên một đoạn chỉ số mà không cần tạo mảng phụ.

In [4]:
print(list(accumulate([3, -1, 2, -4, 0])))
print("min:", min(accumulate([3, -1, 2, -4, 0])))

[3, 2, 4, 0, 0]
min: 0


### 3.4 `bisect` để đếm số phần tử ≤ / ≥ một giá trị

Trên list đã sắp xếp, `bisect_right(a, x)` = số phần tử `≤ x`; `len(a) − bisect_left(a, x)` = số phần tử `≥ x`.

In [5]:
a = [1, 3, 3, 5, 8]
x = 3
print("số phần tử <= 3:", bisect.bisect_right(a, x), "| số phần tử >= 3:", len(a) - bisect.bisect_left(a, x))

số phần tử <= 3: 3 | số phần tử >= 3: 4


## 4. Ý tưởng nền tảng / trực giác

Cố định tổng đích `T`. Với cặp `(a, b)` (sắp sao cho `a ≤ b`):
- **0 bước** nếu `a + b = T`.
- **1 bước** nếu đổi **một** số là đủ: giữ `a`, đổi `b` thành `T − a ∈ [1, limit]` ⇒ `T ∈ [a + 1, a + limit]`; hoặc giữ `b`, đổi `a` ⇒ `T ∈ [b + 1, b + limit]`. Hợp hai đoạn (chúng chồng nhau vì `a ≤ b ≤ limit`) là `[a + 1, b + limit]`.
- **2 bước** trong mọi trường hợp còn lại: `T ∈ [2, a]` hoặc `T ∈ [b + limit + 1, 2·limit]` (đổi cả hai số thì đạt được mọi `T ∈ [2, 2·limit]`).

Tính tay ví dụ 1: `limit = 4`, các cặp `(1, 3)` và `(2, 4)`.

| T | cặp (1,3) | cặp (2,4) | tổng |
|---|---|---|---|
| 2 | 2 | 2 | 4 |
| 3 | 1 | 2 | 3 |
| 4 | **0** | 1 | **1** |
| 5 | 1 | 1 | 2 |
| 6 | 1 | **0** | 1 |
| 7 | 1 | 1 | 2 |
| 8 | 2 | 1 | 3 |

⇒ đáp án `min = 1` (tại `T = 4` hoặc `T = 6`).

In [6]:
def pair_cost(a, b, T, limit):
    if a > b:
        a, b = b, a
    if a + b == T:
        return 0
    if a + 1 <= T <= b + limit:
        return 1
    return 2

nums, limit = [1, 2, 4, 3], 4
P = pairs_of(nums)
print("T :", list(range(2, 2 * limit + 1)))
for a, b in P:
    print(f"{(a, b)}:", [pair_cost(a, b, T, limit) for T in range(2, 2 * limit + 1)])
print("tổng:", [sum(pair_cost(a, b, T, limit) for a, b in P) for T in range(2, 2 * limit + 1)])

T : [2, 3, 4, 5, 6, 7, 8]
(1, 3): [1, 1, 0, 1, 1, 1, 2]
(2, 4): [2, 1, 1, 1, 0, 1, 1]
tổng: [3, 2, 1, 2, 1, 2, 3]


## 5. Cách 1 — Brute Force / baseline

### 5.1 Vét cạn theo định nghĩa: thử mọi mảng đích

Thử **mọi** mảng `t` với `t[i] ∈ [1, limit]`, giữ các mảng bổ sung, đếm số vị trí khác `nums`. Chỉ chạy được với `n`, `limit` rất nhỏ (`limitⁿ` mảng) — dùng làm **chuẩn đối chiếu**.

In [7]:
def min_moves_definition(nums, limit):
    best = len(nums)
    for t in product(range(1, limit + 1), repeat=len(nums)):
        if is_complementary(t):
            best = min(best, sum(x != y for x, y in zip(nums, t)))
    return best

for nums, limit, exp in examples:
    print(min_moves_definition(nums, limit), "(kỳ vọng", exp, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


### 5.2 Thử mọi tổng đích `T`

Với mỗi `T ∈ [2, 2·limit]`, cộng `pair_cost` của mọi cặp. Đúng nhờ phân tích ở mục 4.

In [8]:
def min_moves_try_T(nums, limit):
    P = pairs_of(nums)
    return min(sum(pair_cost(a, b, T, limit) for a, b in P) for T in range(2, 2 * limit + 1))

for nums, limit, exp in examples:
    print(min_moves_try_T(nums, limit), "(kỳ vọng", exp, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


**Độ phức tạp:** cách 5.1 `O(limitⁿ · n)`; cách 5.2 `O(limit · n)` — với `limit = n = 10⁵` là `5·10⁹` phép tính: quá chậm.

**Nút thắt:** với mỗi cặp, chi phí theo `T` chỉ là một **hàm bậc thang** thay đổi giá trị tại **4 điểm**. Ta đang tính lại nó ở **mọi** `T`.

## 6. Ý tưởng thuật toán / bước đột phá

Chi phí cặp `(a, b)` (với `a ≤ b`) theo `T` trên `[2, 2·limit]`:

```
T:     2 ........ a | a+1 ...... a+b−1 | a+b | a+b+1 ...... b+limit | b+limit+1 ...... 2·limit
cost:  2             | 1                 |  0  | 1                    | 2
```

Thay vì tính từng `T`, ta ghi các **điểm thay đổi** vào mảng hiệu `D` (chỉ số theo `T`):
- Toàn đoạn `[2, 2·limit]`: `+2` ⇒ `D[2] += 2`.
- Từ `a + 1`: giảm 1 (2 → 1) ⇒ `D[a + 1] −= 1`.
- Tại `a + b`: giảm thêm 1 (1 → 0) ⇒ `D[a + b] −= 1`; từ `a + b + 1` tăng lại 1 ⇒ `D[a + b + 1] += 1`.
- Từ `b + limit + 1`: tăng 1 (1 → 2) ⇒ `D[b + limit + 1] += 1`.

Sau khi ghi cho mọi cặp, tổng tiền tố `S[T] = Σ D[2..T]` là **tổng chi phí** với tổng đích `T`. Đáp án = `min_{T ∈ [2, 2·limit]} S[T]`.

**Bất biến:** sau khi xử lý `k` cặp, tổng tiền tố của `D` tại mọi `T` bằng tổng chi phí của `k` cặp đó với `T`.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Mảng hiệu (lời giải chính)

In [9]:
def min_moves_diff(nums, limit):
    n = len(nums)
    D = [0] * (2 * limit + 2)            # chỉ số tới b + limit + 1 <= 2*limit + 1
    for i in range(n // 2):
        a, b = nums[i], nums[n - 1 - i]
        if a > b:
            a, b = b, a
        D[2] += 2
        D[a + 1] -= 1
        D[a + b] -= 1
        D[a + b + 1] += 1
        D[b + limit + 1] += 1
    best, cur = float('inf'), 0
    for T in range(2, 2 * limit + 1):       # D[0] = D[1] = 0 nên bắt đầu cộng từ T = 2
        cur += D[T]
        best = min(best, cur)
    return best

for nums, limit, exp in examples:
    print(min_moves_diff(nums, limit), "(kỳ vọng", exp, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


Viết gọn hơn: `D[0] = D[1] = 0` nên tổng tiền tố từ chỉ số `2` chính là `accumulate(D[2:])`; chỉ cần lấy `min` trên `T ∈ [2, 2·limit]` (bỏ phần tử cuối ứng với chỉ số `2·limit + 1`).

In [10]:
class SolutionDiff:
    def minMoves(self, nums: List[int], limit: int) -> int:
        n = len(nums)
        D = [0] * (2 * limit + 2)
        for i in range(n // 2):
            a, b = nums[i], nums[n - 1 - i]
            if a > b:
                a, b = b, a
            D[2] += 2
            D[a + 1] -= 1
            D[a + b] -= 1
            D[a + b + 1] += 1
            D[b + limit + 1] += 1
        return min(accumulate(D[2:2 * limit + 1]))

print([SolutionDiff().minMoves(nums, limit) for nums, limit, _ in examples])

[1, 2, 0]


**Độ phức tạp:** `O(n + limit)` thời gian, `O(limit)` bộ nhớ.

### 7.2 Cách 3 — Sweep line với sự kiện đã sắp xếp

Khi `limit` rất lớn (ví dụ `10⁹`) mà `n` nhỏ, mảng hiệu cỡ `2·limit` không khả thi. Ta chỉ lưu **các sự kiện** `(vị trí, thay đổi)` — tối đa `4` sự kiện mỗi cặp — sắp xếp và quét. Giá trị chỉ đổi tại các điểm sự kiện, nên chỉ cần xét `T` tại những điểm đó (và điểm đầu `2`).

In [11]:
def min_moves_sweep(nums, limit):
    n = len(nums)
    m = n // 2
    events = {}
    def add(pos, v):
        events[pos] = events.get(pos, 0) + v
    for i in range(m):
        a, b = sorted((nums[i], nums[n - 1 - i]))
        add(a + 1, -1)
        add(a + b, -1)
        add(a + b + 1, +1)
        add(b + limit + 1, +1)
    cur = 2 * m                            # chi phí tại T = 2 trước khi áp sự kiện tại 2 (không có, vì a + 1 >= 2 và a + b >= 2)
    best = float('inf')
    for pos in sorted(events):
        if pos > 2 * limit:
            break
        if pos > 2:
            best = min(best, cur)          # giá trị trên đoạn [điểm trước, pos - 1]
        cur += events[pos]
    best = min(best, cur)                  # đoạn cuối tới 2*limit
    return best

for nums, limit, exp in examples:
    print(min_moves_sweep(nums, limit), "(kỳ vọng", exp, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


Chú ý: tại `T = 2` có thể có sự kiện (`a + 1 = 2` khi `a = 1`, hoặc `a + b = 2`). Ta xét giá trị **sau khi** áp các sự kiện tại mỗi điểm, và giá trị trước sự kiện đầu tiên `> 2` là giá trị trên đoạn bắt đầu từ `2`. **Độ phức tạp:** `O(n log n)` thời gian (sắp xếp `≤ 4·n/2` sự kiện), `O(n)` bộ nhớ — không phụ thuộc `limit`.

### 7.3 Cách 4 — Đếm bằng tìm kiếm nhị phân

Với `m` cặp và tổng đích `T`: tổng chi phí `= 2m − (số cặp có T ∈ [a+1, b+limit]) − (số cặp có a + b = T)`. Đếm số cặp có `a + 1 ≤ T` và `b + limit ≥ T` bằng `bisect` trên hai mảng đã sắp `lo = sorted(a+1)`, `hi = sorted(b+limit)` — vì luôn có `a + 1 ≤ b + limit`, số cặp "chứa" `T` = `#(lo ≤ T) − #(hi < T)`.

In [12]:
from collections import Counter

def min_moves_bisect(nums, limit):
    P = [tuple(sorted(p)) for p in pairs_of(nums)]
    m = len(P)
    lo = sorted(a + 1 for a, b in P)
    hi = sorted(b + limit for a, b in P)
    same = Counter(a + b for a, b in P)
    best = float('inf')
    for T in range(2, 2 * limit + 1):
        inside = bisect.bisect_right(lo, T) - bisect.bisect_left(hi, T)
        best = min(best, 2 * m - inside - same.get(T, 0))
    return best

for nums, limit, exp in examples:
    print(min_moves_bisect(nums, limit), "(kỳ vọng", exp, ")")

1 (kỳ vọng 1 )
2 (kỳ vọng 2 )
0 (kỳ vọng 0 )


`bisect_left(hi, T)` = số cặp có `b + limit < T` (những cặp đã "ra khỏi" đoạn 1 bước). **Độ phức tạp:** `O(n log n + limit · log n)` thời gian, `O(n)` bộ nhớ. Có thể chỉ xét `T` tại các điểm ứng viên (các `a + b`, `a + 1`, `2`) để bỏ phụ thuộc `limit`.

## 8. TRACE từng bước

Trace cách 2 trên ví dụ 1: `nums = [1,2,4,3]`, `limit = 4`. Hai cặp `(1, 3)` và `(2, 4)`.

| cặp (a, b) | D[2] | D[a+1] | D[a+b] | D[a+b+1] | D[b+limit+1] |
|---|---|---|---|---|---|
| (1, 3) | +2 | D[2] −1 | D[4] −1 | D[5] +1 | D[8] +1 |
| (2, 4) | +2 | D[3] −1 | D[6] −1 | D[7] +1 | D[9] +1 |

Tổng tiền tố từ `T = 2`: `[3, 2, 1, 2, 1, 2, 3]` cho `T = 2..8` ⇒ min = `1`. (Khớp bảng tính tay ở mục 4.)

In [13]:
def trace_diff(nums, limit):
    n = len(nums)
    D = [0] * (2 * limit + 2)
    for i in range(n // 2):
        a, b = sorted((nums[i], nums[n - 1 - i]))
        D[2] += 2; D[a + 1] -= 1; D[a + b] -= 1; D[a + b + 1] += 1; D[b + limit + 1] += 1
        print(f"cặp ({a},{b}): D[2]+=2, D[{a + 1}]-=1, D[{a + b}]-=1, D[{a + b + 1}]+=1, D[{b + limit + 1}]+=1 -> D={D}")
    S = list(accumulate(D[2:2 * limit + 1]))
    for T, v in zip(range(2, 2 * limit + 1), S):
        print(f"   T={T}: tổng chi phí = {v}")
    print("min =", min(S))
    return min(S)

trace_diff([1, 2, 4, 3], 4)

cặp (1,3): D[2]+=2, D[2]-=1, D[4]-=1, D[5]+=1, D[8]+=1 -> D=[0, 0, 1, 0, -1, 1, 0, 0, 1, 0]
cặp (2,4): D[2]+=2, D[3]-=1, D[6]-=1, D[7]+=1, D[9]+=1 -> D=[0, 0, 3, -1, -1, 1, -1, 1, 1, 1]
   T=2: tổng chi phí = 3
   T=3: tổng chi phí = 2
   T=4: tổng chi phí = 1
   T=5: tổng chi phí = 2
   T=6: tổng chi phí = 1
   T=7: tổng chi phí = 2
   T=8: tổng chi phí = 3
min = 1


1

In [14]:
trace_diff([1, 2, 2, 1], 2)

cặp (1,1): D[2]+=2, D[2]-=1, D[2]-=1, D[3]+=1, D[4]+=1 -> D=[0, 0, 0, 1, 1, 0]
cặp (2,2): D[2]+=2, D[3]-=1, D[4]-=1, D[5]+=1, D[5]+=1 -> D=[0, 0, 2, 0, 0, 2]
   T=2: tổng chi phí = 2
   T=3: tổng chi phí = 2
   T=4: tổng chi phí = 2
min = 2


2

Trace cách 3 (sweep) trên ví dụ 1 để thấy chỉ cần xét các điểm sự kiện:

In [15]:
def trace_sweep(nums, limit):
    n = len(nums); m = n // 2
    events = {}
    for i in range(m):
        a, b = sorted((nums[i], nums[n - 1 - i]))
        for pos, v in ((a + 1, -1), (a + b, -1), (a + b + 1, 1), (b + limit + 1, 1)):
            events[pos] = events.get(pos, 0) + v
    print("sự kiện:", dict(sorted(events.items())))
    cur, best, prev = 2 * m, float('inf'), 2
    for pos in sorted(events):
        if pos > 2 * limit:
            break
        if pos > 2:
            print(f"   đoạn T ∈ [{prev}, {pos - 1}]: chi phí {cur}")
            best = min(best, cur)
        cur += events[pos]; prev = max(prev, pos)
    print(f"   đoạn T ∈ [{prev}, {2 * limit}]: chi phí {cur}")
    best = min(best, cur)
    print("min =", best)
    return best

trace_sweep([1, 2, 4, 3], 4)

sự kiện: {2: -1, 3: -1, 4: -1, 5: 1, 6: -1, 7: 1, 8: 1, 9: 1}
   đoạn T ∈ [2, 2]: chi phí 3
   đoạn T ∈ [3, 3]: chi phí 2
   đoạn T ∈ [4, 4]: chi phí 1
   đoạn T ∈ [5, 5]: chi phí 2
   đoạn T ∈ [6, 6]: chi phí 1
   đoạn T ∈ [7, 7]: chi phí 2
   đoạn T ∈ [8, 8]: chi phí 3
min = 1


1

## 9. Code nộp LeetCode

In [16]:
class Solution:
    def minMoves(self, nums: List[int], limit: int) -> int:
        n = len(nums)
        D = [0] * (2 * limit + 2)             # D[T]: thay đổi chi phí khi T tăng qua giá trị này
        for i in range(n // 2):
            a, b = nums[i], nums[n - 1 - i]
            if a > b:
                a, b = b, a
            D[2] += 2                         # mặc định mỗi cặp tốn 2
            D[a + 1] -= 1                     # từ a+1: chỉ cần 1 bước
            D[a + b] -= 1                     # đúng a+b: 0 bước
            D[a + b + 1] += 1                 # sau a+b: lại 1 bước
            D[b + limit + 1] += 1             # sau b+limit: lại 2 bước
        best, cur = float('inf'), 0
        for T in range(2, 2 * limit + 1):
            cur += D[T]
            if cur < best:
                best = cur
        return best

for nums, limit, exp in examples:
    assert Solution().minMoves(nums, limit) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Quy về tổng đích.** Mảng sau khi sửa là bổ sung với một tổng chung `T`; vì mọi phần tử trong `[1, limit]` nên `T ∈ [2, 2·limit]`. Các cặp độc lập (mỗi phần tử thuộc đúng một cặp), nên với `T` cố định, số bước tối thiểu = `Σ cost(cặp, T)`. Đáp án = `min_T Σ cost(cặp, T)`.

**Chi phí một cặp đúng như phân tích.** Với `a ≤ b`:
- `cost = 0` ⇔ không đổi gì ⇔ `a + b = T`.
- `cost ≤ 1` ⇔ đổi đúng một số: giữ `a` cần `T − a ∈ [1, limit]` ⇔ `T ∈ [a+1, a+limit]`; giữ `b` cần `T ∈ [b+1, b+limit]`. Vì `a ≤ b ≤ limit` ⇒ `b + 1 ≤ a + limit + 1`, hai đoạn liền nhau/chồng nhau ⇒ hợp là `[a+1, b+limit]`.
- `cost = 2` luôn đạt được với mọi `T ∈ [2, 2·limit]` (chọn hai số trong `[1, limit]` có tổng `T`).

**Mảng hiệu tính đúng tổng.** Hàm `cost(T)` của một cặp bằng `2 − [T ≥ a+1] − [T ≥ a+b] + [T ≥ a+b+1] + [T ≥ b+limit+1]` trên `[2, 2·limit]` (kiểm tra trên từng đoạn ở mục 6). Mỗi hạng tử `[T ≥ p]` chính là tổng tiền tố của một đơn vị đặt tại `D[p]`. Cộng mọi cặp ⇒ tổng tiền tố của `D` tại `T` là `Σ cost`. Lưu ý thứ tự `a + 1 ≤ a + b` (vì `b ≥ 1`) và `a + b + 1 ≤ b + limit + 1` (vì `a ≤ limit`) nên các đoạn không bị "đảo".

## 11. Phân tích Time / Space Complexity

Gọi `m = n / 2` là số cặp, `L = limit`.

| Cách | Time | Space |
|---|---|---|
| 1a. Thử mọi mảng đích | `O(Lⁿ · n)` | `O(n)` |
| 1b. Thử mọi `T` | `O(m · L)` | `O(m)` |
| 2. Mảng hiệu | `O(m + L)` | `O(L)` |
| 3. Sweep line sự kiện | `O(m log m)` | `O(m)` |
| 4. Đếm bằng `bisect` | `O(m log m + L log m)` | `O(m)` |

In [17]:
rnd = random.Random(1674)
L = 10**5
big = [rnd.randint(1, L) for _ in range(10**5)]
for name, fn in [("diff", min_moves_diff), ("SolDiff", SolutionDiff().minMoves), ("sweep", min_moves_sweep), ("bisect", min_moves_bisect), ("Solution", Solution().minMoves)]:
    t0 = time.perf_counter(); r = fn(big, L); dt = time.perf_counter() - t0
    print(f"{name:<9}: {r:>6}  {dt * 1000:7.1f} ms")
Ls = 300
small = [rnd.randint(1, Ls) for _ in range(400)]
t0 = time.perf_counter(); r = min_moves_try_T(small, Ls); dt = time.perf_counter() - t0
print(f"try_T n=400, limit=300: {r} ({dt * 1000:.0f} ms) | diff: {min_moves_diff(small, Ls)}")

diff     :  49997     37.8 ms
SolDiff  :  49997     25.8 ms


sweep    :  49997     75.7 ms


bisect   :  49997    103.5 ms


Solution :  49997     21.5 ms
try_T n=400, limit=300: 197 (13 ms) | diff: 197


## 12. Bộ test

In [18]:
ALL = {
    "try_T": min_moves_try_T,
    "diff": min_moves_diff,
    "sweep": min_moves_sweep,
    "bisect": min_moves_bisect,
    "SolDiff": lambda a, l: SolutionDiff().minMoves(a, l),
    "Solution": lambda a, l: Solution().minMoves(a, l),
}

def check(nums, limit, exp=None, definition=False):
    ref = min_moves_definition(nums, limit) if definition else min_moves_try_T(nums, limit)
    for name, fn in ALL.items():
        got = fn(list(nums), limit)
        assert got == ref, (name, nums, limit, got, ref)
    if exp is not None:
        assert ref == exp, (nums, limit, ref, exp)
    return ref

for nums_, limit_, exp in examples:
    check(nums_, limit_, exp, definition=True)
print("Ví dụ đề (cả vét cạn theo định nghĩa): OK")

check([1, 1], 1, 0)
check([1, 2], 2, 0)
check([1, 1, 2, 2], 2, 0)                  # cặp (1,2),(1,2): cùng tổng 3
check([1, 1, 1, 1], 5, 0)
check([1, 5, 5, 1], 5, 2)                  # cặp (1,1),(5,5): chọn T=6, mỗi cặp đổi 1 số
check([1, 1, 5, 5], 5, 0)                  # cặp (1,5),(1,5): cùng tổng 6
check([1, 1, 1, 5], 5, 1)                  # cặp (1,5),(1,1): đổi một số
print("Edge case: OK")

Ví dụ đề (cả vét cạn theo định nghĩa): OK
Edge case: OK


In [19]:
rnd = random.Random(2015)
for _ in range(400):
    n = rnd.choice([2, 4])
    limit = rnd.randint(1, 4 if n == 4 else 6)
    nums_ = [rnd.randint(1, limit) for _ in range(n)]
    check(nums_, limit, definition=True)
print("400 bộ rất nhỏ so với vét cạn theo định nghĩa: OK")

for _ in range(1500):
    n = 2 * rnd.randint(1, 12)
    limit = rnd.randint(1, 15)
    nums_ = [rnd.randint(1, limit) for _ in range(n)]
    check(nums_, limit)
print("1500 bộ ngẫu nhiên nhỏ so với thử mọi T: OK")

for _ in range(10):
    n = 2 * rnd.randint(1, 5000)
    limit = rnd.randint(1, 10**5)
    nums_ = [rnd.randint(1, limit) for _ in range(n)]
    a = Solution().minMoves(nums_, limit)
    assert a == min_moves_sweep(nums_, limit) == SolutionDiff().minMoves(nums_, limit)
print("10 bộ cỡ lớn so giữa các cách nhanh: OK")

400 bộ rất nhỏ so với vét cạn theo định nghĩa: OK


1500 bộ ngẫu nhiên nhỏ so với thử mọi T: OK


10 bộ cỡ lớn so giữa các cách nhanh: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ thử các tổng `T` đang có sẵn (tổng xuất hiện nhiều nhất)

In [20]:
def min_moves_bug_existing_sums(nums, limit):
    P = pairs_of(nums)
    cands = {a + b for a, b in P}                         # SAI: chỉ xét T là tổng có sẵn
    return min(sum(pair_cost(a, b, T, limit) for a, b in P) for T in cands)

rnd = random.Random(3)
found = None
while found is None:
    n = 2 * rnd.randint(2, 4); limit = rnd.randint(2, 6)
    nums_ = [rnd.randint(1, limit) for _ in range(n)]
    if min_moves_bug_existing_sums(nums_, limit) != Solution().minMoves(nums_, limit):
        found = (nums_, limit)
nums_, limit_ = found
print("phản ví dụ:", found, "| chỉ xét tổng có sẵn:", min_moves_bug_existing_sums(nums_, limit_), "| đúng:", Solution().minMoves(nums_, limit_))
P = pairs_of(nums_)
best_T = min(range(2, 2 * limit_ + 1), key=lambda T: sum(pair_cost(a, b, T, limit_) for a, b in P))
print("   các cặp:", P, "| tổng có sẵn:", sorted({a + b for a, b in P}), "| T tốt nhất:", best_T)

phản ví dụ: ([2, 1, 4, 4, 3, 4, 1, 1], 4) | chỉ xét tổng có sẵn: 5 | đúng: 4
   các cặp: [(2, 1), (1, 1), (4, 4), (4, 3)] | tổng có sẵn: [2, 3, 7, 8] | T tốt nhất: 5


**Vì sao sai:** đôi khi một tổng **chưa cặp nào có** lại cho phép **mọi** cặp chỉ tốn 1 bước, tốt hơn việc giữ nguyên một cặp (0 bước) nhưng bắt các cặp khác tốn 2. **Sửa:** xét **mọi** `T ∈ [2, 2·limit]` (mảng hiệu làm việc này với chi phí thấp).

### Lỗi 2 — Lấy `min` trên cả các chỉ số `T < 2`

In [21]:
def min_moves_bug_range(nums, limit):
    n = len(nums)
    D = [0] * (2 * limit + 2)
    for i in range(n // 2):
        a, b = sorted((nums[i], nums[n - 1 - i]))
        D[2] += 2; D[a + 1] -= 1; D[a + b] -= 1; D[a + b + 1] += 1; D[b + limit + 1] += 1
    return min(accumulate(D))                      # SAI: gồm T = 0, 1 (chi phí 0 giả)

nums_, limit_ = [1, 2, 4, 3], 4
print("sai :", min_moves_bug_range(nums_, limit_), "| đúng:", Solution().minMoves(nums_, limit_))

sai : 0 | đúng: 1


**Vì sao sai:** `D[0] = D[1] = 0` nên tổng tiền tố tại `T = 0, 1` bằng 0 — nhưng `T = 0, 1` **không đạt được** (mỗi số `≥ 1`). Tương tự, chỉ số `2·limit + 1` cũng ngoài phạm vi. **Sửa:** chỉ lấy `min` trên `T ∈ [2, 2·limit]`.

### Lỗi 3 — Không sắp `a ≤ b` trước khi đánh dấu

In [22]:
def min_moves_bug_no_sort(nums, limit):
    n = len(nums)
    D = [0] * (2 * limit + 2)
    for i in range(n // 2):
        a, b = nums[i], nums[n - 1 - i]              # SAI: quên đổi để a <= b
        D[2] += 2; D[a + 1] -= 1; D[a + b] -= 1; D[a + b + 1] += 1
        if b + limit + 1 < len(D):
            D[b + limit + 1] += 1
    return min(accumulate(D[2:2 * limit + 1]))

rnd = random.Random(8)
found = None
while found is None:
    n = 2 * rnd.randint(1, 3); limit = rnd.randint(2, 6)
    nums_ = [rnd.randint(1, limit) for _ in range(n)]
    if min_moves_bug_no_sort(nums_, limit) != Solution().minMoves(nums_, limit):
        found = (nums_, limit)
nums_, limit_ = found
print("phản ví dụ:", found, "| các cặp:", pairs_of(nums_))
print("sai :", min_moves_bug_no_sort(nums_, limit_), "| đúng:", Solution().minMoves(nums_, limit_))

phản ví dụ: ([2, 2, 2, 2, 2, 1], 2) | các cặp: [(2, 1), (2, 2), (2, 2)]
sai : 2 | đúng: 1


**Vì sao sai:** đoạn 1 bước là `[min + 1, max + limit]`. Nếu `a > b`, ta đánh dấu `[a + 1, b + limit]` — đoạn **hẹp hơn** thực tế (thậm chí có thể rỗng/đảo), làm chi phí bị tính quá cao. **Sửa:** `if a > b: a, b = b, a`.

### Lỗi 4 — Mảng hiệu thiếu một ô

In [23]:
def min_moves_bug_size(nums, limit):
    n = len(nums)
    D = [0] * (2 * limit + 1)                          # SAI: cần 2*limit + 2
    for i in range(n // 2):
        a, b = sorted((nums[i], nums[n - 1 - i]))
        D[2] += 2; D[a + 1] -= 1; D[a + b] -= 1; D[a + b + 1] += 1; D[b + limit + 1] += 1
    return min(accumulate(D[2:]))

try:
    print(min_moves_bug_size([2, 2], 2))
except IndexError as e:
    print("IndexError:", e, "| đúng:", Solution().minMoves([2, 2], 2))

IndexError: list index out of range | đúng: 0


**Vì sao sai:** với `b = limit`, chỉ số `b + limit + 1 = 2·limit + 1` — cần mảng dài `2·limit + 2`; tương tự `a + b + 1` có thể bằng `2·limit + 1`. **Sửa:** cấp phát `2·limit + 2` ô.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Mọi cặp/phần tử phải đạt **cùng một giá trị** `T`" ⇒ **liệt kê `T`** và tính tổng chi phí; nếu chi phí từng phần là **hàm bậc thang** của `T` ⇒ **mảng hiệu**.
- Mảng hiệu: "cộng `v` cho đoạn `[l, r]`" ⇔ `D[l] += v; D[r+1] −= v`. Kích thước = vị trí lớn nhất + 2.
- Miền `T` quá lớn ⇒ **sweep line** trên các điểm sự kiện (sắp xếp).
- Luôn kiểm tra miền hợp lệ của `T` khi lấy `min`/`max`.
- Viết bảng chi phí `cặp × T` cho ví dụ nhỏ để kiểm công thức bậc thang.

## 15. Biến thể của bài

1. **Chi phí đổi khác nhau** (ví dụ đổi phần tử lớn tốn hơn): vẫn là hàm bậc thang theo `T` nếu chi phí chỉ phụ thuộc số phần tử bị đổi.
2. **Thay vì đổi tùy ý, chỉ được tăng/giảm 1 mỗi bước** (chi phí `|thay đổi|`): chi phí mỗi cặp là hàm **tuyến tính từng khúc** theo `T` ⇒ dùng mảng hiệu bậc hai hoặc tìm trung vị.
3. **Yêu cầu hiệu `nums[i] − nums[n−1−i]` bằng nhau:** phân tích tương tự với đoạn khác.
4. **LeetCode 1094 — Car Pooling / 370 — Range Addition:** mảng hiệu cơ bản.
5. **LeetCode 2381 — Shifting Letters II:** mảng hiệu trên chuỗi.
6. **Đếm có bao nhiêu `T` đạt chi phí tối thiểu:** đếm trên tổng tiền tố.

In [24]:
def all_best_T(nums, limit):
    n = len(nums)
    D = [0] * (2 * limit + 2)
    for i in range(n // 2):
        a, b = sorted((nums[i], nums[n - 1 - i]))
        D[2] += 2; D[a + 1] -= 1; D[a + b] -= 1; D[a + b + 1] += 1; D[b + limit + 1] += 1
    S = list(accumulate(D[2:2 * limit + 1]))
    best = min(S)
    return best, [T for T, v in zip(range(2, 2 * limit + 1), S) if v == best]

def range_addition(length, updates):
    """LeetCode 370: áp các cập nhật (l, r, v) bằng mảng hiệu."""
    D = [0] * (length + 1)
    for l, r, v in updates:
        D[l] += v; D[r + 1] -= v
    return list(accumulate(D))[:length]

print("ví dụ 1: chi phí tối thiểu và mọi T đạt được:", all_best_T([1, 2, 4, 3], 4))
print("370:", range_addition(5, [(1, 3, 2), (2, 4, 3), (0, 2, -2)]))

ví dụ 1: chi phí tối thiểu và mọi T đạt được: (1, [4, 6])
370: [-2, 0, 3, 5, 3]


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Mọi mảng đích | đúng theo định nghĩa | `O(Lⁿ n)` | `O(n)` | kiểm thử cực nhỏ |
| 1b. Mọi `T` | cộng `pair_cost` | `O(m L)` | `O(m)` | kiểm thử, hiểu bài |
| 2. Mảng hiệu | bậc thang 0/1/2 ⇒ 5 cập nhật/cặp | `O(m + L)` | `O(L)` | **nộp bài** |
| 3. Sweep line | sắp các điểm sự kiện | `O(m log m)` | `O(m)` | `L` rất lớn |
| 4. Đếm `bisect` | `2m − inside − same` | `O(m log m + L log m)` | `O(m)` | cách nghĩ "đếm" trực quan |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `Solution().minMoves([1, 3, 3, 1], 3)` bằng bao nhiêu?

**Bài 2 (tính tay):** Lập bảng `cost(cặp, T)` cho `nums = [1, 4, 2, 3]`, `limit = 4` (các cặp `(1, 3)`, `(4, 2)`), tìm mọi `T` tối ưu.

**Bài 3 (điền TODO):** Hoàn thiện `best_target_and_array(nums, limit)` trả về `(số bước tối thiểu, T, mảng sau khi sửa)` — mảng sau khi sửa phải bổ sung với tổng `T`, mọi phần tử trong `[1, limit]`, và khác `nums` đúng ở số vị trí bằng số bước.

**Bài 4 (tự viết lại):** Viết lại cách 4 nhưng chỉ xét `T` thuộc tập ứng viên `{2} ∪ {a+1} ∪ {a+b} ∪ {a+b+1} ∪ {b+limit+1}` (lọc trong `[2, 2·limit]`), độ phức tạp `O(m log m)`.

**Bài 5 (mini challenge):** Biến thể chi phí "mỗi bước tăng/giảm 1 đơn vị" (đổi `x` thành `y` tốn `|x − y|`, vẫn trong `[1, limit]`). Viết `min_cost_unit_steps(nums, limit)` — gợi ý: với `T` cố định, chi phí một cặp là `max(0, |T − (a + b)|)` nếu giới hạn cho phép… hãy kiểm bằng vét cạn nhỏ.

In [25]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def best_target_and_array(nums, limit):
    # TODO: tìm T tối ưu bằng mảng hiệu; với mỗi cặp: 0 bước nếu đã bằng T; 1 bước: giữ số phù hợp, đổi số kia;
    #       2 bước: chọn x = max(1, T - limit), y = T - x
    pass

# Bài 4 — TODO
def min_moves_candidates(nums, limit):
    # TODO: như min_moves_bisect nhưng vòng lặp T chỉ chạy trên tập ứng viên
    pass

# Bài 5 — TODO
def min_cost_unit_steps(nums, limit):
    # TODO: chi phí cặp (a, b) để đạt T = |T - (a+b)| nếu T trong [2, 2*limit] (luôn chia được phần thay đổi cho hai số)
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1**
- Mức 1: Ghép `nums[i]` với `nums[n−1−i]`: các cặp là `(1, 1)` và `(3, 3)` (không phải `(1, 3)`!).
- Mức 2: Tổng có sẵn là `2` và `6`; giữ một tổng thì cặp kia phải đổi **cả hai** số.
- Mức 3: Chọn `T = 4`: mỗi cặp chỉ đổi 1 số ⇒ đáp án `2`.

**Bài 2**
- Mức 1: Cặp `(1, 3)`: 0 bước tại `T = 4`, 1 bước trên `[2, 7]`.
- Mức 2: Cặp `(2, 4)`: 0 bước tại `T = 6`, 1 bước trên `[3, 8]`.
- Mức 3: `T = 4` và `T = 6` đều cho tổng 1.

**Bài 3**
- Mức 1: Dùng `all_best_T` để lấy một `T`.
- Mức 2: Với 1 bước: nếu `1 ≤ T − a ≤ limit` thì giữ `a`, ngược lại giữ `b`.
- Mức 3: Kiểm tra bằng `is_complementary` và đếm số vị trí khác.

**Bài 4**
- Mức 1: Giá trị chỉ đổi tại các điểm sự kiện.
- Mức 2: Mỗi đoạn hằng bắt đầu tại một điểm ứng viên.
- Mức 3: Đừng quên điểm `2`.

**Bài 5**
- Mức 1: Muốn tổng cặp từ `a + b` thành `T`, tổng thay đổi ít nhất là `|T − (a + b)|`.
- Mức 2: Luôn chia được lượng thay đổi cho hai số mà vẫn trong `[1, limit]` khi `T ∈ [2, 2·limit]`.
- Mức 3: Đáp án là `min_T Σ |T − s_i|` ⇒ `T` là **trung vị** của các tổng `s_i`.

## 19. Đáp án tham khảo cho bài tập

In [26]:
# Bài 1
print("Bài 1:", Solution().minMoves([1, 3, 3, 1], 3))
assert Solution().minMoves([1, 3, 3, 1], 3) == 2 == min_moves_definition([1, 3, 3, 1], 3)

# Bài 2
nums2, lim2 = [1, 4, 2, 3], 4
P2 = pairs_of(nums2)
for a, b in P2:
    print(f"  {(a, b)}:", [pair_cost(a, b, T, lim2) for T in range(2, 2 * lim2 + 1)])
print("  Bài 2:", all_best_T(nums2, lim2))
assert all_best_T(nums2, lim2) == (1, [4, 6])

# Bài 3
def best_target_and_array_sol(nums, limit):
    best, Ts = all_best_T(nums, limit)
    T = Ts[0]
    n = len(nums)
    out = list(nums)
    for i in range(n // 2):
        a, b = nums[i], nums[n - 1 - i]
        if a + b == T:
            continue
        if 1 <= T - a <= limit:
            out[n - 1 - i] = T - a
        elif 1 <= T - b <= limit:
            out[i] = T - b
        else:
            x = max(1, T - limit)
            out[i], out[n - 1 - i] = x, T - x
    return best, T, out

rnd = random.Random(3)
for _ in range(2000):
    n = 2 * rnd.randint(1, 8); limit = rnd.randint(1, 10)
    nums_ = [rnd.randint(1, limit) for _ in range(n)]
    best, T, out = best_target_and_array_sol(nums_, limit)
    assert is_complementary(out) and all(1 <= v <= limit for v in out)
    assert sum(x != y for x, y in zip(nums_, out)) == best == Solution().minMoves(nums_, limit)
print("Bài 3 OK | ví dụ 1:", best_target_and_array_sol([1, 2, 4, 3], 4))

Bài 1: 2
  (1, 3): [1, 1, 0, 1, 1, 1, 2]
  (4, 2): [2, 1, 1, 1, 0, 1, 1]
  Bài 2: (1, [4, 6])
Bài 3 OK | ví dụ 1: (1, 4, [1, 2, 2, 3])


In [27]:
# Bài 4
def min_moves_candidates_sol(nums, limit):
    P = [tuple(sorted(p)) for p in pairs_of(nums)]
    m = len(P)
    lo = sorted(a + 1 for a, b in P)
    hi = sorted(b + limit for a, b in P)
    same = Counter(a + b for a, b in P)
    cand = {2}
    for a, b in P:
        cand.update((a + 1, a + b, a + b + 1, b + limit + 1))
    best = float('inf')
    for T in cand:
        if 2 <= T <= 2 * limit:
            inside = bisect.bisect_right(lo, T) - bisect.bisect_left(hi, T)
            best = min(best, 2 * m - inside - same.get(T, 0))
    return best

rnd = random.Random(4)
for _ in range(2000):
    n = 2 * rnd.randint(1, 10); limit = rnd.randint(1, 20)
    nums_ = [rnd.randint(1, limit) for _ in range(n)]
    assert min_moves_candidates_sol(nums_, limit) == Solution().minMoves(nums_, limit)
print("Bài 4 OK")

# Bài 5
def min_cost_unit_steps_sol(nums, limit):
    sums = sorted(a + b for a, b in pairs_of(nums))
    med = sums[len(sums) // 2]
    return sum(abs(s - med) for s in sums)

def min_cost_unit_brute(nums, limit):
    best = None
    for t in product(range(1, limit + 1), repeat=len(nums)):
        if is_complementary(t):
            c = sum(abs(x - y) for x, y in zip(nums, t))
            best = c if best is None else min(best, c)
    return best

rnd = random.Random(5)
for _ in range(300):
    n = rnd.choice([2, 4]); limit = rnd.randint(1, 5)
    nums_ = [rnd.randint(1, limit) for _ in range(n)]
    assert min_cost_unit_steps_sol(nums_, limit) == min_cost_unit_brute(nums_, limit), (nums_, limit)
print("Bài 5 OK | [1,2,4,3], limit=4 ->", min_cost_unit_steps_sol([1, 2, 4, 3], 4))

Bài 4 OK
Bài 5 OK | [1,2,4,3], limit=4 -> 2


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "chọn mục tiêu chung T, chi phí từng phần là bậc thang" bằng mảng hiệu:**

```python
D = [0] * (MAX_T + 2)
for part in parts:
    for (start, delta) in breakpoints(part):     # chi phí tăng/giảm delta kể từ T = start
        D[start] += delta
best, cur = float('inf'), 0
for T in range(MIN_T, MAX_T + 1):
    cur += D[T]
    best = min(best, cur)
```

**Bài này:** với `a ≤ b`: `D[2] += 2; D[a+1] −= 1; D[a+b] −= 1; D[a+b+1] += 1; D[b+limit+1] += 1`, `T ∈ [2, 2·limit]`.

**Checklist:**
- [ ] Sắp `a ≤ b` cho từng cặp?
- [ ] Đoạn 1 bước là `[a + 1, b + limit]`?
- [ ] Mảng hiệu dài `2·limit + 2`?
- [ ] Chỉ lấy `min` trên `T ∈ [2, 2·limit]`?
- [ ] Xét **mọi** `T`, không chỉ các tổng có sẵn?
- [ ] Đã so với "thử mọi T" và vét cạn định nghĩa trên input nhỏ?
- [ ] Chữ ký đúng: `def minMoves(self, nums: List[int], limit: int) -> int`?